# reshape-back — worked example 3: reshape_back through a two-step reshape chain

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reshape-back`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When a forward chains two reshapes, the reverse pass calls `reshape_back` once per step in reverse order, each time targeting the shape that step's input had. Because reshape is its own inverse family, every call is just `grad_out.reshape(input_shape)`.

## Worked solution

The forward goes `(6, 4) -> (24,) -> (2, 12)`. To backprop a gradient on the final `(2, 12)` output, we walk backward: first `reshape_back` from `(2, 12)` to the intermediate `(24,)`, then from `(24,)` to the original `(6, 4)`. Each call reshapes to the shape the corresponding forward input had. The composed result equals reshaping the final gradient straight to `(6, 4)`, which we confirm against autograd on the full chain.

In [ ]:
Tensor = t.Tensor


def reshape_back(grad_out: Tensor, out: Tensor, x: Tensor, new_shape: tuple) -> Tensor:
    return grad_out.reshape(x.shape)


def chain_back(grad_out, x_leaf):
    u = x_leaf.reshape(24)          # (6,4) -> (24,)
    y = u.reshape(2, 12)            # (24,) -> (2,12)
    g_u = reshape_back(grad_out, y, u, (2, 12))
    g_x = reshape_back(g_u, u, x_leaf, (24,))
    return g_x


t.manual_seed(2)
x = t.randn(6, 4)
grad_out = t.randn(2, 12)
grad_x = chain_back(grad_out, x)
print('grad_x shape:', tuple(grad_x.shape))

xg = x.clone().requires_grad_(True)
xg.reshape(24).reshape(2, 12).backward(grad_out)
print('matches autograd:', t.allclose(grad_x, xg.grad))